# OctoTetrahedral AGI - Multi-Modal Demo + ARC Training
**Genus-13 | 160M params | Vision + Audio + Embodiment**

Runtime > Run all. One shot. ~10 min total.

In [ ]:
# CELL 1: Setup everything
!rm -rf /content/octo
!git clone --depth 1 https://github.com/GitMonsters/octotetrahedral-agi.git /content/octo
%cd /content/octo
!pip install -q tiktoken pyyaml tqdm numpy Pillow
!rm -rf arc_data && git clone --depth 1 https://github.com/fchollet/ARC-AGI.git arc_data 2>/dev/null
import glob
train_files = sorted(glob.glob('arc_data/data/training/*.json'))
eval_files = sorted(glob.glob('arc_data/data/evaluation/*.json'))
print(f'[OK] Repo cloned | ARC-AGI: {len(train_files)} train + {len(eval_files)} eval tasks')

In [ ]:
# CELL 2: GPU + Load model ONCE
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'[GPU] {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB)')
else:
    print('[CPU] Enable GPU: Runtime > Change runtime type > T4')
print(f'PyTorch: {torch.__version__}')

from config import Config
from model import OctoTetrahedralModel
cfg = Config()
cfg.cognitive_geometry.entropy_monitor_enabled = False
cfg.cognitive_geometry.svd_enabled = False
model = OctoTetrahedralModel(cfg, use_geometric_physics=False)
model = model.to(device)
params = sum(p.numel() for p in model.parameters())
print(f'[MODEL] {params/1e6:.1f}M params on {device} | VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB')

In [ ]:
# CELL 3: Test all modalities
import time

def make_input(**kw):
    d = {'input_ids': torch.randint(0, 100, (1, 64)).to(device)}
    for k, v in kw.items():
        d[k] = v.to(device=device, dtype=torch.float32)
    return d

tests = {
    'Text only': {},
    'Text + Vision': {'images': torch.randn(1, 3, 64, 64)},
    'Text + Audio': {'audio': torch.randn(1, 16000)},
    'Text + Embodiment': {'proprioception': torch.randn(1, 32), 'touch': torch.randn(1, 64)},
    'ALL MODALITIES': {'images': torch.randn(1, 3, 64, 64), 'audio': torch.randn(1, 16000), 'proprioception': torch.randn(1, 32), 'touch': torch.randn(1, 64)},
}

print('\nMulti-Modal Forward Pass Tests')
print('=' * 60)
for name, extra in tests.items():
    inp = make_input(**extra)
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model(**inp)
    if device == 'cuda': torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) * 1000
    mm = out.get('multimodal_info', {})
    fused = 'YES' if mm.get('fused') else 'no'
    print(f'  {name:25s} | {ms:7.1f}ms | fused: {fused} | modalities: {mm.get("num_modalities", 0)}')
print('\n[OK] All modalities working')

In [ ]:
# CELL 4: Vision encoder test
grid = [[i % 10 for i in range(5)] for _ in range(5)]
vis_result = model.vision_encoder.encode_grid(grid)
print(f'Vision encoded: {vis_result["embeddings"].shape}')
print('[OK] ARC grid -> vision pipeline working')

In [ ]:
# CELL 5: Architecture summary
from core.distributed_scale import ModelParallelismManager
from config import ScaleConfig

print('\nScale Presets:')
print('=' * 65)
for preset in ['tiny', 'base', 'large', 'ultra']:
    sc = ScaleConfig(preset=preset)
    hw = sc.get_hardware_estimate()
    marker = ' <-- YOU ARE HERE' if preset == 'tiny' else ''
    print(f"  {preset:6s} | {hw['total_params_str']:>8} total | {hw['active_params_str']:>8} active | {hw['min_h100_gpus']:>4} H100s{marker}")

print(f'\nCurrent: {params/1e6:.0f}M params on {device}')
print(f'Target: 1.75T params (ultra preset, 512+ H100 GPUs)')
print(f'Genus-13 topology: 14 compound braid streams')

---
## ARC-AGI Training

In [ ]:
# CELL 6: ARC Training - pure FP32, no autocast, no scaler
import torch.nn.functional as F
import numpy as np
import json, random

torch.cuda.empty_cache()

NUM_EPOCHS = 3
BATCH_SIZE = 1
LR = 1e-4
MAX_GRID = 8
NUM_TASKS = 100
steps_per_epoch = NUM_TASKS

def grid_to_tensor(grid, max_size=MAX_GRID):
    h, w = len(grid), len(grid[0])
    flat = [v for row in grid for v in row][:max_size*max_size]
    flat = flat + [0] * max(0, max_size*max_size - len(flat))
    tokens = torch.tensor(flat, dtype=torch.long)
    colors = torch.tensor([[0,0,0],[0,0,1],[1,0,0],[0,1,0],[1,1,0],
        [.5,.5,.5],[1,0,1],[1,.5,0],[0,1,1],[.5,0,0]])
    img = torch.zeros(3, max_size, max_size)
    for r in range(min(h, max_size)):
        for c in range(min(w, max_size)):
            img[:, r, c] = colors[grid[r][c]]
    return tokens, img

def load_arc_sample(files):
    f = random.choice(files)
    task = json.load(open(f))
    pair = random.choice(task['train'])
    it, ii = grid_to_tensor(pair['input'])
    ot, _ = grid_to_tensor(pair['output'])
    return (it.unsqueeze(0).to(device),
            ii.unsqueeze(0).to(device, dtype=torch.float32),
            ot.unsqueeze(0).to(device))

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
history = []
best_loss = float('inf')
t0 = time.time()

print(f'[TRAIN] {NUM_TASKS} tasks | {NUM_EPOCHS} epochs | batch {BATCH_SIZE} | grid {MAX_GRID}x{MAX_GRID} | {steps_per_epoch * NUM_EPOCHS} total steps')
print(f'[VRAM] {torch.cuda.memory_allocated()/1e9:.1f}GB used')
print()

for epoch in range(NUM_EPOCHS):
    ep_loss = []
    for step in range(steps_per_epoch):
        ids, imgs, tgt = load_arc_sample(train_files[:NUM_TASKS])
        optimizer.zero_grad(set_to_none=True)
        out = model(input_ids=ids, images=imgs)
        logits = out['logits']
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=0)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        l = loss.item()
        ep_loss.append(l)
        history.append(l)
        if step % 20 == 0:
            print(f'  E{epoch+1}/{NUM_EPOCHS} | Step {step:3d}/{steps_per_epoch} | Loss: {l:.4f} | VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB | {time.time()-t0:.0f}s')
        del ids, imgs, tgt, out, logits, loss
        torch.cuda.empty_cache()
    avg = np.mean(ep_loss)
    if avg < best_loss:
        best_loss = avg
        torch.save(model.state_dict(), 'arc_multimodal_best.pt')
    print(f'\n  --- Epoch {epoch+1} done | Avg: {avg:.4f} | Best: {best_loss:.4f} ---\n')

t_total = time.time() - t0
print(f'[DONE] {t_total:.0f}s | Loss: {history[0]:.4f} -> {history[-1]:.4f} ({(1-history[-1]/history[0])*100:.1f}% reduction)')
print(f'[SAVE] arc_multimodal_best.pt')

In [ ]:
# CELL 7: Loss plot
import matplotlib.pyplot as plt

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
a1.plot(history, alpha=0.3, color='blue', label='Step loss')
w = min(20, len(history)//3)
if w > 1:
    s = np.convolve(history, np.ones(w)/w, mode='valid')
    a1.plot(range(w-1, len(history)), s, color='red', lw=2, label=f'Smoothed ({w}-step)')
a1.set_xlabel('Step'); a1.set_ylabel('Loss')
a1.set_title('OctoTetrahedral ARC Training Loss')
a1.legend(); a1.grid(True, alpha=0.3)
epoch_avgs = [np.mean(history[i*steps_per_epoch:(i+1)*steps_per_epoch]) for i in range(NUM_EPOCHS)]
a2.bar(range(1, NUM_EPOCHS+1), epoch_avgs, color=['#ff6b6b','#ffd93d','#6bcb77'][:NUM_EPOCHS])
a2.set_xlabel('Epoch'); a2.set_ylabel('Avg Loss')
a2.set_title('Loss by Epoch')
a2.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()
print('[OK] Plot done')

In [ ]:
# CELL 8: Evaluate on held-out tasks
model.eval()
correct = 0
total = 0
print('Evaluating on 30 held-out ARC tasks...')
with torch.no_grad():
    for f in eval_files[:30]:
        task = json.load(open(f))
        for pair in task['test']:
            in_tok, in_img = grid_to_tensor(pair['input'])
            out_tok, _ = grid_to_tensor(pair['output'])
            in_tok = in_tok.unsqueeze(0).to(device)
            in_img = in_img.unsqueeze(0).to(device, dtype=torch.float32)
            out_tok = out_tok.to(device)
            result = model(input_ids=in_tok, images=in_img)
            pred = result['logits'][0].argmax(dim=-1)
            mask = out_tok != 0
            if mask.sum() > 0:
                acc = (pred[mask] == out_tok[mask]).float().mean().item()
                if acc > 0.95:
                    correct += 1
            total += 1
print(f'\nResults: {correct}/{total} correct (>95% pixel acc)')
print(f'Score: {correct/max(total,1)*100:.1f}%')
print(f'\nNote: {NUM_EPOCHS} epochs on {NUM_TASKS} tasks, grid {MAX_GRID}x{MAX_GRID}.')
print(f'Full training + program synthesis = production scores.')